In [2]:
import numpy as np
import pandas as pd
# import data_clean_utils
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, MinMaxScaler, PowerTransformer, OrdinalEncoder
from sklearn.model_selection import train_test_split

In [3]:
from sklearn import set_config

set_config(transform_output="pandas")

# Load the Data

In [4]:
# load the data

df = pd.read_csv('swiggy.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'swiggy.csv'

# Clean Data

In [ ]:
data_clean_utils.perform_data_cleaning(df)

In [ ]:
# load the cleaned data

df = pd.read_csv('swiggy_cleaned.csv')

df

In [ ]:
df.columns

In [ ]:
# drop columns not required for model input

columns_to_drop =  ['rider_id',
                    'restaurant_latitude',
                    'restaurant_longitude',
                    'delivery_latitude',
                    'delivery_longitude',
                    'order_date',
                    "order_time_hour",
                    "order_day"]

df.drop(columns=columns_to_drop, inplace=True)

df

In [ ]:
# check for missing values

df.isna().sum()

In [ ]:
# check for duplicates

df.duplicated().sum()

In [ ]:
import missingno as msno

msno.matrix(df)

In [ ]:
# columns that have missing values

missing_cols = (
                    df
                    .isna()
                    .any(axis=0)
                    .loc[lambda x: x]
                    .index
                )

missing_cols

# Data Preparation

In [ ]:
temp_df = df.copy().dropna()

In [ ]:
# split into X and y

X = temp_df.drop(columns='time_taken')
y = temp_df['time_taken']

X

In [ ]:
# train test split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
print("The size of train data is",X_train.shape)
print("The shape of test data is",X_test.shape)

In [ ]:
y_train

In [ ]:
# missing data in training data

X_train.isna().sum()

In [ ]:
X_train.columns

In [ ]:
len(X_train.columns)

In [ ]:
# do basic preprocessing

num_cols = ["age","ratings","pickup_time_minutes","distance"]

nominal_cat_cols = ['weather','type_of_order',
                    'type_of_vehicle',"festival",
                    "city_type","city_name","order_month",
                    "order_day_of_week",
                    "is_weekend",
                    "order_time_of_day"]

ordinal_cat_cols = ["traffic","distance_type"]

In [ ]:
len(num_cols + nominal_cat_cols + ordinal_cat_cols)

In [ ]:
for col in ordinal_cat_cols:
    print(col,X_train[col].unique())

In [ ]:
# generate order for ordinal encoding

traffic_order = ["low","medium","high","jam"]

distance_type_order = ["short","medium","long","very_long"]

In [ ]:
# build a preprocessor

preprocessor = ColumnTransformer(transformers=[
    ("scale", MinMaxScaler(), num_cols),
    ("nominal_encode", OneHotEncoder(drop="first",handle_unknown="ignore",sparse_output=False), nominal_cat_cols),
    ("ordinal_encode", OrdinalEncoder(categories=[traffic_order,distance_type_order]), ordinal_cat_cols)
],remainder="passthrough",n_jobs=-1,force_int_remainder_cols=False,verbose_feature_names_out=False)

preprocessor.set_output(transform="pandas")

In [ ]:
# transform the data

X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)

X_train_trans

In [ ]:
# transform target column

pt = PowerTransformer()

y_train_pt = pt.fit_transform(y_train.values.reshape(-1,1))
y_test_pt = pt.transform(y_test.values.reshape(-1,1))

In [ ]:
pt.lambdas_

In [ ]:
y_train_pt

# Train Initial Baseline Model

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()

lr.fit(X_train_trans,y_train_pt)

In [ ]:
# get the predictions
y_pred_train = lr.predict(X_train_trans)
y_pred_test = lr.predict(X_test_trans)

In [ ]:
# get the actual predictions values

y_pred_train_org = pt.inverse_transform(y_pred_train.reshape(-1,1))
y_pred_test_org = pt.inverse_transform(y_pred_test.reshape(-1,1))

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

print(f"The train error is {mean_absolute_error(y_train,y_pred_train_org):.2f} minutes")
print(f"The test error is {mean_absolute_error(y_test,y_pred_test_org):.2f} minutes")

In [ ]:
print(f"The train r2 score is {r2_score(y_train,y_pred_train_org):.2f}")
print(f"The test r2 score is {r2_score(y_test,y_pred_test_org):.2f}")

# Impute missing values

In [ ]:
temp_df = df.copy()

In [ ]:
# split into X and y

X = temp_df.drop(columns='time_taken')
y = temp_df['time_taken']

X

In [ ]:
# train test split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
print("The size of train data is",X_train.shape)
print("The shape of test data is",X_test.shape)

In [ ]:
# missing values in train data

X_train.isna().sum()

In [ ]:
# transform target column

pt = PowerTransformer()

y_train_pt = pt.fit_transform(y_train.values.reshape(-1,1))
y_test_pt = pt.transform(y_test.values.reshape(-1,1))

In [ ]:
missing_cols

In [ ]:
# percentage of rows in data having missing values

(
    X_train
    .isna()
    .any(axis=1)
    .mean()
    .round(2) * 100
)


# Age

In [ ]:
X_train['age'].describe()

In [ ]:
# missing values in the column

X_train['age'].isna().sum()

In [ ]:
# median value

age_median = X_train['age'].median()

**Avg and Median values are similar, impute the age column with median value**

In [ ]:
# plot the kde plot

sns.kdeplot(X_train['age'],label="original")
sns.kdeplot(X_train['age'].fillna(age_median),label="imputed")
plt.legend()

**Observation**:

1. Changed the distribution of the age column.
2. Use Advanced imputation techniques like KNN imputer.

## Ratings

In [ ]:
# statistical summary

X_train['ratings'].describe()

In [ ]:
# missing values

X_train['ratings'].isna().sum()

In [ ]:
# avg rating

ratings_mean = X_train['ratings'].mean()

# fill and plot kdeplot

sns.kdeplot(X_train['ratings'],label="original")
sns.kdeplot(X_train['ratings'].fillna(ratings_mean),label="imputed")
plt.legend()

# Weather

In [ ]:
# value counts

X_train['weather'].value_counts()

In [ ]:
# missing values in the column

X_train['weather'].isna().sum()

In [ ]:
# countplot

sns.countplot(X_train['weather'])

**No dominant category to impute from**


In [ ]:
# capture the missingness

missing_weather = MissingIndicator()
missing_weather.set_output(transform="pandas")

pd.concat([X_train['weather'],missing_weather.fit_transform(X_train[['weather']])],axis=1).sample(50)


# Traffic

In [ ]:
# value counts

X_train['traffic'].value_counts()

In [ ]:
# Missing values in column

X_train['traffic'].isna().sum()

In [ ]:
# countplot

sns.countplot(X_train['traffic'])

No Dominant category

In [ ]:
missing_cols

# Multiple Deliveries

In [ ]:
# value counts

X_train['multiple_deliveries'].value_counts()

In [ ]:
# countplot

sns.countplot(X_train['multiple_deliveries'].apply(str))

In [ ]:
# number of missing values

X_train['multiple_deliveries'].isna().sum()

In [ ]:
# mode value

multiple_deliveries_mode = X_train['multiple_deliveries'].mode()[0]

In [ ]:
# fill na values with mode

sns.countplot(X_train['multiple_deliveries'].fillna(multiple_deliveries_mode).apply(str))

# Festival

In [ ]:
# value counts

X_train['festival'].value_counts()

In [ ]:
# countplot

sns.countplot(X_train['festival'])

In [ ]:
# missing values in column

X_train['festival'].isna().sum()

In [ ]:
# mode value

festival_mode = X_train['festival'].mode()[0]

In [ ]:
# fill with mode

sns.countplot(X_train['festival'].fillna(festival_mode))

# City type

In [ ]:
# value counts

X_train['city_type'].value_counts()

In [ ]:
# number of missing values

X_train['city_type'].isna().sum()

In [ ]:
# countplot

sns.countplot(X_train['city_type'])

In [ ]:
# mode value

city_type_mode = X_train['city_type'].mode()[0]

In [ ]:
# fill with mode

sns.countplot(X_train['city_type'].fillna(city_type_mode))

In [ ]:
missing_cols

# Pickup time minutes

In [ ]:
# statistical summary

X_train['pickup_time_minutes'].describe()

In [ ]:
# missing values in the column

X_train['pickup_time_minutes'].isna().sum()

In [ ]:
# median value

pickup_time_minutes_median = X_train['pickup_time_minutes'].median()

In [ ]:
# histplot

sns.histplot(X_train['pickup_time_minutes'],kde=True,label='original')
sns.histplot(X_train['pickup_time_minutes'].fillna(pickup_time_minutes_median),kde=True,label='imputed')
plt.legend()

# Order time of day

In [ ]:
# value counts

X_train['order_time_of_day'].value_counts()

In [ ]:
# missing values

X_train['order_time_of_day'].isna().sum()

In [ ]:
# countplot

sns.countplot(X_train['order_time_of_day'])

In [ ]:
# rows where the data is missing

X_train[X_train['order_time_of_day'].isna()]

# Distance

In [ ]:
# statistical summary

X_train['distance'].describe()

In [ ]:
# number of missing values

X_train['distance'].isna().sum()

In [ ]:
# avg distance

distance_mean = X_train['distance'].mean()

In [ ]:
# kdeplot

sns.kdeplot(X_train['distance'],label='original')
sns.kdeplot(X_train['distance'].fillna(distance_mean),label='imputed')
plt.legend()

In [ ]:
missing_cols

# Distance Type


In [ ]:
# value counts

X_train['distance_type'].value_counts()

In [ ]:
# missing values

X_train['distance_type'].isna().sum()

In [ ]:
# countplot

sns.countplot(X_train['distance_type'])

Mode cannot be used here

# Imputation Pipeline

In [ ]:
nominal_cat_cols

In [ ]:
X_train.isna().sum()

In [ ]:
# features to fill values with mode

features_to_fill_mode = ['multiple_deliveries','festival','city_type']
features_to_fill_missing = [col for col in nominal_cat_cols if col not in features_to_fill_mode]

features_to_fill_missing

In [ ]:
# simple imputer to fill categorical vars with mode

simple_imputer = ColumnTransformer(transformers=[
    ("mode_imputer",SimpleImputer(strategy="most_frequent"),features_to_fill_mode),
    ("missing_imputer",SimpleImputer(strategy="constant",fill_value="missing"),features_to_fill_missing)
],remainder="passthrough",n_jobs=-1,force_int_remainder_cols=False,verbose_feature_names_out=False)

simple_imputer

In [ ]:
simple_imputer.fit_transform(X_train)

In [ ]:
simple_imputer.fit_transform(X_train).isna().sum()

In [ ]:
# knn imputer

knn_imputer = KNNImputer(n_neighbors=5)

In [ ]:
# do basic preprocessing

num_cols = ["age","ratings","pickup_time_minutes","distance"]

nominal_cat_cols = ['weather','type_of_order',
                    'type_of_vehicle',"festival",
                    "city_type","city_name","order_month",
                    "order_day_of_week",
                    "is_weekend",
                    "order_time_of_day"]

ordinal_cat_cols = ["traffic","distance_type"]

In [ ]:
# generate order for ordinal encoding

traffic_order = ["low","medium","high","jam"]

distance_type_order = ["short","medium","long","very_long"]

In [ ]:
# unique categories the ordinal columns

for col in ordinal_cat_cols:
    print(col,X_train[col].unique())

In [ ]:
# build a preprocessor

preprocessor = ColumnTransformer(transformers=[
    ("scale", MinMaxScaler(), num_cols),
    ("nominal_encode", OneHotEncoder(drop="first",handle_unknown="ignore",
                                     sparse_output=False), nominal_cat_cols),
    ("ordinal_encode", OrdinalEncoder(categories=[traffic_order,distance_type_order],
                                      encoded_missing_value=-999,
                                      handle_unknown="use_encoded_value",
                                      unknown_value=-1), ordinal_cat_cols)
],remainder="passthrough",n_jobs=-1,force_int_remainder_cols=False,verbose_feature_names_out=False)


preprocessor

In [ ]:
preprocessor.fit_transform(X_train)

In [ ]:
preprocessor.fit_transform(X_train).isna().sum().loc[lambda ser : ser.ge(1)]

In [ ]:
# build the pipeline

processing_pipeline = Pipeline(steps=[
                                ("simple_imputer",simple_imputer),
                                ("preprocess",preprocessor),
                                ("knn_imputer",knn_imputer)
                            ])

processing_pipeline

In [ ]:
# fit and transform the pipeline on X_train

processing_pipeline.fit_transform(X_train)

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()

model_pipe = Pipeline(steps=[
                                ("preprocessing",processing_pipeline),
                                ("model",lr)
                            ])

model_pipe

In [ ]:
# fit the pipeline on data

model_pipe.fit(X_train,y_train_pt)

In [ ]:
# get the predictions
y_pred_train = model_pipe.predict(X_train)
y_pred_test = model_pipe.predict(X_test)

In [ ]:
# get the actual predictions values

y_pred_train_org = pt.inverse_transform(y_pred_train.reshape(-1,1))
y_pred_test_org = pt.inverse_transform(y_pred_test.reshape(-1,1))

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

print(f"The train error is {mean_absolute_error(y_train,y_pred_train_org):.2f} minutes")
print(f"The test error is {mean_absolute_error(y_test,y_pred_test_org):.2f} minutes")

In [ ]:
print(f"The train r2 score is {r2_score(y_train,y_pred_train_org):.2f}")
print(f"The test r2 score is {r2_score(y_test,y_pred_test_org):.2f}")

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(random_state=42,n_jobs=-1)

model_pipe = Pipeline(steps=[
                                ("preprocessing",processing_pipeline),
                                ("model",rf)
                            ])

model_pipe

In [ ]:
# fit the pipeline on data

model_pipe.fit(X_train,y_train_pt.values.ravel())

In [ ]:
# get the predictions
y_pred_train = model_pipe.predict(X_train)
y_pred_test = model_pipe.predict(X_test)

In [ ]:
# get the actual predictions values

y_pred_train_org = pt.inverse_transform(y_pred_train.reshape(-1,1))
y_pred_test_org = pt.inverse_transform(y_pred_test.reshape(-1,1))

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

print(f"The train error is {mean_absolute_error(y_train,y_pred_train_org):.2f} minutes")
print(f"The test error is {mean_absolute_error(y_test,y_pred_test_org):.2f} minutes")

In [ ]:
print(f"The train r2 score is {r2_score(y_train,y_pred_train_org):.2f}")
print(f"The test r2 score is {r2_score(y_test,y_pred_test_org):.2f}")